## RETESTE COM AMOSTRA REAL SINAN-TB
Fonte: `dados_tuberculose_v2_10000.csv` (9.999 registros). **Não é uma reprodução integral do estudo.**

Pré-processamento restrito às variáveis necessárias e desfechos 1/2. Ordem pela data de notificação, não apenas ano. Modelo demonstrativo SGDClassifier, não River.

# Capítulo 21 — versão executável demonstrativa

**Atenção:** os blocos originais são reproduzidos abaixo como texto (não executados), para evitar substituir silenciosamente `river`/`deep_river`. A demonstração usa `sklearn.linear_model.SGDClassifier`, que **não é** Hoeffding Tree nem Deep River. Todos os dados são sintéticos.

In [1]:
import importlib.util
for p in ["river", "deep_river", "torch"]:
    print(p, "disponível" if importlib.util.find_spec(p) else "INDISPONÍVEL")

river INDISPONÍVEL
deep_river INDISPONÍVEL
torch disponível


### Bloco de Código 21.1 — Classificação incremental com uma Hoeffding Tree

*Página 437 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
from river import tree, metrics

# Hoeffding Tree para classificação
model = tree.HoeffdingTreeClassifier()
metric = metrics.Accuracy()

for xi, yi in data_stream:
    y_pred = model.predict_one(xi)
    metric.update(yi, y_pred)
    model.learn_one(xi, yi)

print(f'Acurácia: {metric.get():.3f}')

```

**Diagnóstico 21.1:** requer `river` e `data_stream`, ambos indisponíveis no ambiente original.

### Bloco de Código 21.2 — Rede neural online com Deep River e PyTorch.

*Página 439 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
from river import metrics, compose, preprocessing
from deep_river.classification import Classifier
from torch import nn

class MyModule(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    
    def forward(self, x):
        return self.net(x)

model = compose.Pipeline(
    preprocessing.StandardScaler(),
    Classifier(module=MyModule, n_features=9)
)

metric = metrics.ROCAUC()
for xi, yi in data_stream:
    y_prob = model.predict_proba_one(xi)
    metric.update(yi, y_prob)
    model.learn_one(xi, yi)

```

**Diagnóstico 21.2:** requer `deep_river`, `river` e `data_stream`. A arquitetura declara 9 atributos, enquanto o bloco 21.4 seleciona 5; são exemplos distintos, não diretamente encadeáveis.

### Bloco de Código 21.3 — Importação e filtragem dos dados do SINAN-TB.

*Página 440 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
import pandas as pd

df = pd.read_csv('dados_tuberculose_v2.csv')

colunas = ['NU_ANO', 'SG_UF_NOT', 'FORMA',
           'RAIOX_TORA', 'IDADE_ANOS', 'SITUA_ENCE']
df = df[colunas].copy()

# Manter apenas cura (1) e abandono (2)
df = df[df['SITUA_ENCE'].isin([1, 2])]

print(f'Total de casos: {len(df)}')

```

**Adaptação para 21.3 e 21.4:** gerar base sintética com o esquema mínimo; os dados não são registros reais do SINAN-TB. Os códigos originais são mantidos acima.

In [2]:
import numpy as np,pandas as pd
base=pd.read_csv(r"/mnt/data/dados_tuberculose_v2_10000.csv",low_memory=False)
print('Base REAL:',base.shape)


Base REAL: (9999, 98)


### Bloco de Código 21.4 — Pré-processamento dos dados para aprendizado online.

*Página 440 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
# Remover linhas com valores ausentes
df = df.dropna()

# Criar variável alvo: 1 = abandono, 0 = cura
df['alvo'] = (df['SITUA_ENCE'] == 2).astype(int)

# Ordenar por ano (simula ordem cronológica)
df = df.sort_values('NU_ANO').reset_index(drop=True)

# Separar features e alvo
features = ['SG_UF_NOT', 'FORMA', 'RAIOX_TORA', 'IDADE_ANOS', 'NU_ANO']
X = df[features]
y = df['alvo']

print(f'Cura: {(y==0).sum()} | Abandono: {(y==1).sum()}')

```

In [3]:
df=base.loc[base['SITUA_ENCE'].isin([1,2]), ['SG_UF_NOT','FORMA','RAIOX_TORA','IDADE_ANOS','NU_ANO','DT_NOTIFIC','SITUA_ENCE']].copy()
features=['SG_UF_NOT','FORMA','RAIOX_TORA','IDADE_ANOS','NU_ANO']
for c in features: df[c]=pd.to_numeric(df[c],errors='coerce')
df=df.dropna(subset=features+['SITUA_ENCE']).copy()
df['alvo']=(df['SITUA_ENCE']==2).astype(int)
df['DT_NOTIFIC']=pd.to_datetime(df['DT_NOTIFIC'],errors='coerce')
df=df.sort_values(['DT_NOTIFIC','NU_ANO'],na_position='last').reset_index(drop=True)
X=df[features];y=df['alvo']
print('Cura:',sum(y==0),'Abandono:',sum(y==1),'Total:',len(y))


Cura: 4433 Abandono: 2997 Total: 7430


In [4]:
assert len(X)>1000 and set(y.unique())=={0,1}
print('Validação preparação real:',len(X))


Validação preparação real: 7430


### Bloco de Código 21.5 — Treinamento prequential com uma Hoeffding Tree.

*Página 441 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
from river import tree, metrics

model = tree.HoeffdingTreeClassifier()
metric = metrics.Accuracy()

# Guardar a acurácia a cada 10.000 casos
historico = []

for i in range(len(X)):
    xi = X.iloc[i].to_dict()
    yi = y.iloc[i]

    # Test: prediz antes de aprender
    y_pred = model.predict_one(xi)
    metric.update(yi, y_pred)

    # Train: aprende com o exemplo
    model.learn_one(xi, yi)

    if (i + 1) % 10000 == 0:
        historico.append({
            'exemplos': i + 1,
            'acuracia': metric.get()
        })

print(f'Acurácia final: {metric.get():.3f}')

```

**Adaptação 21.5:** demonstração incremental *test-then-train* com `SGDClassifier.partial_fit` (não é Hoeffding Tree). Escalonamento incremental omitido para preservar simplicidade; variáveis normalizadas por constantes fixas, sem aprender estatísticas no teste.

In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score
from pathlib import Path
model_demo=SGDClassifier(loss='log_loss',random_state=21)
X_demo=X.astype(float).copy()
X_demo['IDADE_ANOS']/=100
X_demo['NU_ANO']=(X_demo['NU_ANO']-2015)/10
X_demo['SG_UF_NOT']/=100
X_demo['FORMA']/=5
X_demo['RAIOX_TORA']/=5
historico=[]; corretos=0; avaliados=0
for i,(xi,yi) in enumerate(zip(X_demo.to_numpy(),y.to_numpy())):
    if i>0:
        pred=int(model_demo.predict(xi.reshape(1,-1))[0])
        corretos+=int(pred==yi); avaliados+=1
    model_demo.partial_fit(xi.reshape(1,-1),np.array([yi]),classes=np.array([0,1]))
    if (i+1)%1000==0: historico.append({'exemplos':i+1,'acuracia':corretos/max(1,avaliados)})
print('Acurácia prequential demonstrativa:',round(corretos/max(1,avaliados),3),'em',avaliados,'predições')
assert avaliados==len(X)-1


Acurácia prequential demonstrativa: 0.582 em 7429 predições


### Bloco de Código 21.6 — Visualização da evolução da acurácia do modelo online.

*Página 442 do PDF · Revisão: confirmado*

**Código original preservado, não executado nesta versão:**

```python
import matplotlib.pyplot as plt

hist = pd.DataFrame(historico)
plt.figure(figsize=(8, 4))
plt.plot(hist['exemplos'], hist['acuracia'])
plt.xlabel('Numero de exemplos vistos')
plt.ylabel('Acuracia acumulada')
plt.title('Evolucao da acuracia - Hoefdding Tree')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figuras/evolucao_acuracia.png', dpi=150)
plt.show()

```

**Adaptação 21.6:** a frequência de histórico é 1.000 amostras, não 10.000, e a pasta de saída é criada explicitamente.

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
Path('figuras').mkdir(exist_ok=True)
hist=pd.DataFrame(historico)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(hist['exemplos'],hist['acuracia'],marker='o')
ax.set(xlabel='Exemplos vistos',ylabel='Acurácia prequential acumulada',title='SGDClassifier incremental — demonstração com amostra real')
ax.grid(alpha=.3)
fig.tight_layout()
fig.savefig('figuras/evolucao_acuracia.png',dpi=130)
plt.show()
assert Path('figuras/evolucao_acuracia.png').exists()
print('Figura salva com',len(hist),'pontos')

Figura salva com 7 pontos


/tmp/ipykernel_9120/3630773844.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
